<a href="https://colab.research.google.com/github/Subroy1/MSAI_AllPracticeModules/blob/main/Module%2011/PracticalAssignement2_used_cars_II.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# What drives the price of a car?

![](images/kurt.jpeg)

**OVERVIEW**

In this application, you will explore a dataset from Kaggle. The original dataset contained information on 3 million used cars. The provided dataset contains information on 426K cars to ensure speed of processing.  Your goal is to understand what factors make a car more or less expensive.  As a result of your analysis, you should provide clear recommendations to your client -- a used car dealership -- as to what consumers value in a used car.

### CRISP-DM Framework

<center>
    <img src = images/crisp.png width = 50%/>
</center>


To frame the task, throughout our practical applications, we will refer back to a standard process in industry for data projects called CRISP-DM.  This process provides a framework for working through a data problem.  Your first step in this application will be to read through a brief overview of CRISP-DM [here](https://mo-pcco.s3.us-east-1.amazonaws.com/BH-PCMLAI/module_11/readings_starter.zip).  After reading the overview, answer the questions below.

### Business Understanding

From a business perspective, we are tasked with identifying key drivers for used car prices.  In the CRISP-DM overview, we are asked to convert this business framing to a data problem definition.  Using a few sentences, reframe the task as a data task with the appropriate technical vocabulary.

<span style="color:blue">**UNDERSTAND THE OBJECTIVE FIRST**</span>

Worldwide, used car sales is a major business occupuation due to its ability to provide affordable transportation to millions of citizens.

Though there is no one-size-fits-all solution as location , region , country , avaiability of public mode of transportation , purpose of owning a car etc and affordability based on demographics also play a part, this exercise makes an effort to understand the fundamental principles of Machine learning algorithm,, specifically LR techniques (Linear and Multiple Linear , Ridge ,Lasso and Elastic Net regression models) in order to price cars based on available features and derived features with additional application of domain knowledge in this field.

Furthermore, since the size of the dataset is huge (400K plus), in order to avoid the model from memorizing the relationships between independent and target variable (price), L1 and L2 regularization and subsequent hyperparamter tuning have been relied upon for creating the optimum model.

During the course of implementing this data mining problem , we would have to constantly refine our solution by not only going back to refine data as needed and repeated remodelling but also by enriching our business understanding from evaluation step.

Finally a summart report would be provided to the intended audience namely used car dealers or franchises who would restructure their inventory accordingly to boost sales.

In [ ]:
# All of the imports needed for the exercise consolidated in the beginning .
#Standard imports
import pandas as pd
import numpy as np

#scikit-learn libs
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler , TargetEncoder, OneHotEncoder, OrdinalEncoder
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.linear_model import Ridge, Lasso, LinearRegression, ElasticNet
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Plotting libs
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

#ignore warnings
import warnings
warnings.filterwarnings('ignore')

**Loading the Data**



In [ ]:
df = pd.read_csv("/content/sample_data/vehicles.csv")
df.head()

,id,region,price,year,manufacturer,model,condition,cylinders,fuel,odometer,title_status,transmission,VIN,drive,size,type,paint_color,state
0,7222695916,prescott,6000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,az
1,7218891961,fayetteville,11900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ar
2,7221797935,florida keys,21000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,fl
3,7222270760,worcester / central MA,1500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ma
4,7210384030,greensboro,4900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nc


# Data understanding the features (Data Dictionary)

*  **price - Our target variable**
*  manufacturer - company manufacturing the brand
*  model - model of the specific make of the car
*  year- year of manufacturing the car or bought by 1st owner
*  transmission - whether manual or automatic gear  
*  drive - whether all wheel drive or 2 wheel drive (Front or rear)
*  state - place where car is registered/ manufactured
*  region- region within the state
*  condition - good or bad (ordinal feature , might need encoding )
*  cylinders- specific to the combustion engine , applicable for petrol and diesel cars only
*   size- such as small , medium , SUV- large , coupe , sedan -medium like that *(might need encoding )
*   fuel- whether runs on gasoline or  diesel , electric (encoding needed)
*   title_status - in which form is the vehicle , it is clean , or rebuilt etc .
* paint_color - car color hardly plays any role in prediction
* odometer- the mileage of the car, how many miles / kms it has driven
* id , VIN - identifiers which do not play any role in predictive algorithms (should be dropped for model building)
* type- utility of the vehicle such as pick up or the structure such as sedan or coupe etc

In [ ]:
# initial random inspection as first few records were NaN for majority of columns in the dataset
df.iloc[150000:150002]

,id,region,price,year,manufacturer,model,condition,cylinders,fuel,odometer,title_status,transmission,VIN,drive,size,type,paint_color,state
150000,7303124206,bloomington,38590,2019.0,ram,1500 crew cab big horn,good,6 cylinders,gas,28556.0,clean,other,1C6RRFFGXKN695394,4wd,NaN,pickup,white,in
150001,7303087652,bloomington,43990,2019.0,ram,1500 crew cab big horn,good,8 cylinders,gas,17352.0,clean,other,1C6SRFFT7KN755503,4wd,NaN,pickup,black,in


Identify numeric and categorical features , also commenting whether we can do any data cleaning or transform into numerical without needing encoder always.

In [ ]:
print(len(df['model'].unique()))
len(df['manufacturer'].unique())

29650


43

In [ ]:
print("title_status",df['title_status'].unique())
print("-----")
print("size",df['size'].unique())# possibly fill with "others" for nan
print("-----")
print("type",df['type'].unique())# possibly fill with "others" for nan
print("-----")
print("cylinders",df['cylinders'].unique())# possibly fill with "others"  # remove cylinders for numerical value # categorical
print("-----")
print("drive",df['drive'].unique())# [nan 'rwd' '4wd' 'fwd'] - we would map them nto numerical quite easily # categorical eventually
print("-----")
print("transmission",df['transmission'].unique()) #transmission [nan 'other' 'automatic' 'manual']  # categorical
print("-----")
print("fuel",df['fuel'].unique()) #  categorical
print("-----")
print("paint_color",df['paint_color'].unique()) # categorical
print("-----")
print("manufacturer",df['manufacturer'].unique()) # categorical  (43)
print("-----")
print("model",df['model'].unique()) #  categorical but 30K , need specialized encoding, will see
print("-----")
print("condition",df['condition'].unique())  #  categorical

title_status [nan 'clean' 'rebuilt' 'lien' 'salvage' 'missing' 'parts only']
-----
size [nan 'full-size' 'mid-size' 'compact' 'sub-compact']
-----
type [nan 'pickup' 'truck' 'other' 'coupe' 'SUV' 'hatchback' 'mini-van' 'sedan'
 'offroad' 'bus' 'van' 'convertible' 'wagon']
-----
cylinders [nan '8 cylinders' '6 cylinders' '4 cylinders' '5 cylinders' 'other'
 '3 cylinders' '10 cylinders' '12 cylinders']
-----
drive [nan 'rwd' '4wd' 'fwd']
-----
transmission [nan 'other' 'automatic' 'manual']
-----
fuel [nan 'gas' 'other' 'diesel' 'hybrid' 'electric']
-----
paint_color [nan 'white' 'blue' 'red' 'black' 'silver' 'grey' 'brown' 'yellow'
 'orange' 'green' 'custom' 'purple']
-----
manufacturer [nan 'gmc' 'chevrolet' 'toyota' 'ford' 'jeep' 'nissan' 'ram' 'mazda'
 'cadillac' 'honda' 'dodge' 'lexus' 'jaguar' 'buick' 'chrysler' 'volvo'
 'audi' 'infiniti' 'lincoln' 'alfa-romeo' 'subaru' 'acura' 'hyundai'
 'mercedes-benz' 'bmw' 'mitsubishi' 'volkswagen' 'porsche' 'kia' 'rover'
 'ferrari' 'mini' 'pon

**Data Preprocessing**

Data Cleaning - Broadly, we will follow the following principle .
1. Any cleaning based on domain knowledge and hence identifying outliers would be done pre train_test_split
2. Any data preprocessing including data cleaning based on statistical properties would be done after train test split to avoid data leakage.





In [ ]:
# Inpsect null values in the dataset , aim to identify null features adding no usability for prediction such as identifiers .
#We observe that many features have loads of null values, # variable with highest number of missing values would appear first.
Total=df.isnull().sum().sort_values(ascending=False)
Total

,0
size,306361
cylinders,177678
condition,174104
VIN,161042
drive,130567
paint_color,130203
type,92858
manufacturer,17646
title_status,8242
model,5277


In [19]:
#since lots of missing values, next calculate percentage of missing values for each variable , sorting descending on pct.
Percent = (df.isnull().sum()*100/df.isnull().count()).sort_values(ascending=False)
# concat the 'Total' and 'Percent' columns using 'concat' function
# pass a list of column names in parameter 'keys'
# 'axis = 1' concats along the columns
missing_data = pd.concat([Total, Percent], axis = 1, keys = ['Total', 'Percentage of Missing Values'])
missing_data


,Total,Percentage of Missing Values
size,306361,71.767476
cylinders,177678,41.622470
condition,174104,40.785232
VIN,161042,37.725356
drive,130567,30.586347
paint_color,130203,30.501078
type,92858,21.752717
manufacturer,17646,4.133714
title_status,8242,1.930753
model,5277,1.236179


In [27]:
# DATA CLEANING
#1. remove id and VIN as they dont add any value to model for predictions .
#df = df.drop(columns={"id","VIN"})

#2. size is missing from 71% , the model and make already indicate what would be the size , hence fair to drop it as it is redundant and would introduce noise and multicollinearity  .
#df = df.drop(columns={"size"})

#3.

In [50]:
# cylinders	177678	  41.622470
# condition	174104	  40.785232
# drive	    130567	  30.586347
# paint_color 130203	30.501078
# type	      92858	  21.752
df['condition'].value_counts(normalize=True) # we will impute condition with good having maximum proportion (48%)
df['cylinders'].value_counts(normalize=True)# will create category 'other' as dsitribution
df['type'].value_counts(normalize=True)# will use 'missing' because no clear majority  class avoid bias
df['paint_color'].value_counts(normalize=True)# will use 'missing' because no clear majority class avoid bias
df['drive'].value_counts(normalize=True)# will group into missing category as no clear winner when just 3 wd types.



,proportion
drive,
4wd,0.445151
fwd,0.356100
rwd,0.198749


In [ ]:
# Visualize the distribution of the target variable price
plt.figure(figsize=(10, 6))
sns.histplot(df['price'], kde=True)
plt.title('Distribution of Target Variable (Car Price)')
plt.xlabel('X')
plt.ylabel('Y')
plt.show()

In [ ]:
# Unique models are

#df['model'].value_counts().sort_values(ascending=False).to_csv('/content/sample_data/model.csv', index=False)
len(df['model'].value_counts())#29649 models
#we need to reduce cardinality , after inspecting the csv above, decided to keep only this models whose value counts >0.5% of total 426K(~ 2100), group all remaining to 'others' category.
#This would ensure model stablity  by reducing sparsity.

#model_counts= df['model'].value_counts()
#model_counts_df= pd.DataFrame(model_counts)


#vehicles_model_df = pd.DataFrame(df['model'].value_counts())
# cap_price=np.quantile(df['price'],0.95)
# df[df['price'] >cap_price].shape

#top_models=np.quantile(vehicles_model_df['count'], 0.95)
#df[df['model']!='f-150'].tail(10)
# encode the missing size to 0





29649

In [ ]:
# Unique states and regions
len(df['state'].unique())#  51 us states - keep
len(df['region'].unique())#  404 us states

404

# **Exploratory Data Analysis**

1.   List item
2.   List item



*Describe the different features within the data*

*Data preprocessing*



**Following CRISP -DM model**
We would first understand the data, describe the statistical properties , observe the categorical and numerical columns (features), define the features for understability .

We will also inspect the quality of the data , null values and any transformations needed.



In [ ]:
df.describe().apply(lambda s: s.apply('{0:.5f}'.format))# Print numerical values , maximum price is abnormal and points to data outliers or errors.
# Since we do not know how many of such values are obviously wrong , we would print the 95 and 99%ile for inspecting further .

,price,year,odometer
count,426880.00000,425675.00000,422480.00000
mean,75199.03319,2011.23519,98043.33144
std,12182282.17360,9.45212,213881.50080
min,0.00000,1900.00000,0.00000
25%,5900.00000,2008.00000,37704.00000
50%,13950.00000,2013.00000,85548.00000
75%,26485.75000,2017.00000,133542.50000
max,3736928711.00000,2022.00000,10000000.00000


In [18]:
# Describe the categorical features
df.describe(include="object")

,region,manufacturer,model,condition,cylinders,fuel,title_status,transmission,VIN,drive,size,type,paint_color,state
count,426880,409234,421603,252776,249202,423867,418638,424324,265838,296313,120519,334022,296677,426880
unique,404,42,29649,6,8,5,6,3,118246,3,4,13,12,51
top,columbus,ford,f-150,good,6 cylinders,gas,clean,automatic,1FMJU1JT1HEA52352,4wd,full-size,sedan,white,ca
freq,3608,70985,8009,121456,94169,356209,405117,336524,261,131904,63465,87056,79285,50614


**TODO**
Plenty of data cleaning scope here , many categorical features and we cannnot discard most of them by inspection, also some of these have very high cardinality, we would need to reduce that smartly .

**TODO**

Distribution plot of target variable (price)- Shows heavily skewed due to erroneous data capture , we would remove these noise and then plot again .

In [ ]:
cap_price=np.quantile(df['price'],0.95)
df[df['price'] >cap_price].shape



(21311, 18)

In [ ]:
# Data preprocessing and cleaning
# Inspect empty values
# First remove features which are blank and count > 70% of the size of the dataset
# Data cleaning- Remove too old data as they were in a different generation
# Remove outliers which are aberrations and belong to 5% beyond range of numericals (beyond 0.95 quantile) - PLot the price or target price , log(price) to show normal distribution
# Plot the corelation matrix and heatmap, plot the linear relationship supposedly between different features and target variable.
np.percentile(df['price'], 99)
# Explore categorical data and create features using encoders.
#target_names=

In [ ]:
#df.tail(2)
print(df['size'].unique())
print("-----")
print(df['drive'].unique())

[nan 'full-size' 'mid-size' 'compact' 'sub-compact']
-----
[nan 'rwd' '4wd' 'fwd']


In [ ]:
df['log_price'] = np.log10(df['price'])

In [ ]:
df.head(2)

,region,price,year,manufacturer,model,condition,cylinders,fuel,odometer,title_status,transmission,drive,size,type,paint_color,state,log_price
0,prescott,6000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,az,3.778151
1,fayetteville,11900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ar,4.075547


In [ ]:
pd.set_option('display.float_format', lambda x: '%.3f' % x)
print(df.describe())
pd.reset_option('display.float_format')

In [ ]:
nitems =  len(df['price'])
df['price'].sort_values(ascending=False).iloc[:20]

In [ ]:
# clearly , the max price is an abberation and hence an outlier to be filtered out (3736928711.000) .
import plotly.express as px
px.histogram(x=df['price'], nbins=5)

### Data Understanding

After considering the business understanding, we want to get familiar with our data.  Write down some steps that you would take to get to know the dataset and identify any quality issues within.  Take time to get to know the dataset and explore what information it contains and how this could be used to inform your business understanding.

Identify numeric and categorical features , also commenting whether we can do any data cleaning or transform into numerical without needing encoder always.

In [ ]:
print("title_status",df['title_status'].unique())
print("-----")
print("size",df['size'].unique())# possibly fill with "others" for nan
print("-----")
print("type",df['type'].unique())# possibly fill with "others" for nan
print("-----")
print("cylinders",df['cylinders'].unique())# possibly fill with "others"  # remove cylinders for numerical value # categorical
print("-----")
print("drive",df['drive'].unique())# [nan 'rwd' '4wd' 'fwd'] - we would map them nto numerical quite easily # categorical eventually
print("-----")
print("transmission",df['transmission'].unique()) #transmission [nan 'other' 'automatic' 'manual']  # categorical
print("-----")
print("fuel",df['fuel'].unique()) #  categorical
print("-----")
print("paint_color",df['paint_color'].unique()) # categorical
print("-----")
print("manufacturer",df['manufacturer'].unique()) # categorical  (43)
print("-----")
print("model",df['model'].unique()) #  categorical but 30K , need specialized encoding, will see
print("-----")
print("condition",df['condition'].unique())  #  categorical

title_status [nan 'clean' 'rebuilt' 'lien' 'salvage' 'missing' 'parts only']
-----
size [nan 'full-size' 'mid-size' 'compact' 'sub-compact']
-----
type [nan 'pickup' 'truck' 'other' 'coupe' 'SUV' 'hatchback' 'mini-van' 'sedan'
 'offroad' 'bus' 'van' 'convertible' 'wagon']
-----
cylinders [nan '8 cylinders' '6 cylinders' '4 cylinders' '5 cylinders' 'other'
 '3 cylinders' '10 cylinders' '12 cylinders']
-----
drive [nan 'rwd' '4wd' 'fwd']
-----
transmission [nan 'other' 'automatic' 'manual']
-----
fuel [nan 'gas' 'other' 'diesel' 'hybrid' 'electric']
-----
paint_color [nan 'white' 'blue' 'red' 'black' 'silver' 'grey' 'brown' 'yellow'
 'orange' 'green' 'custom' 'purple']
-----
manufacturer [nan 'gmc' 'chevrolet' 'toyota' 'ford' 'jeep' 'nissan' 'ram' 'mazda'
 'cadillac' 'honda' 'dodge' 'lexus' 'jaguar' 'buick' 'chrysler' 'volvo'
 'audi' 'infiniti' 'lincoln' 'alfa-romeo' 'subaru' 'acura' 'hyundai'
 'mercedes-benz' 'bmw' 'mitsubishi' 'volkswagen' 'porsche' 'kia' 'rover'
 'ferrari' 'mini' 'pon

In [ ]:
print(len(df['model'].unique())) ## 30K models , cannot be created dummies or encoded , need to trim down differently
len(df['manufacturer'].unique())##43 makes ,ok for now .

29650


43

**Data Preprocessing**

Data Cleaning - Broadly, we will follow the following principle .
1. Any cleaning based on domain knowledge and hence identifying outliers would be done pre train_test_split
2. Any data preprocessing including data cleaning based on statistical properties would be done after train test split to avoid data leakage.





In [ ]:
# Inpsect null values in the dataset , aim to identify null features adding no usability for prediction such as identifiers .
#We observe that many features have loads of null values, # variable with highest number of missing values would appear first.
Total=df.isnull().sum().sort_values(ascending=False)
Total

,0
size,306361
cylinders,177678
condition,174104
VIN,161042
drive,130567
paint_color,130203
type,92858
manufacturer,17646
title_status,8242
model,5277


### Data Preparation

After our initial exploration and fine-tuning of the business understanding, it is time to construct our final dataset prior to modeling.  Here, we want to make sure to handle any integrity issues and cleaning, the engineering of new features, any transformations that we believe should happen (scaling, logarithms, normalization, etc.), and general preparation for modeling with `sklearn`.

In [ ]:
#We will take the apporach of removing the feature completely whereby they have no role to play # in pricing vehicles and loads of them are null .
# for eg - VIN is a vehicle identification number and has nothing to do with car pricing .
df=df.drop(columns={"VIN","id"})

In [ ]:
# REAL DATA CLEANING , ALL ARE NULL -  no statistical loss to model because mandatory features missing from dataset
df = df[~ (df['model'].isnull() & df['year'].isnull()  &  df['condition'].isnull() & df['year'].isnull() & df['fuel'].isnull() & df['odometer'].isnull() & df['title_status'].isnull() & \
   df['transmission'].isnull() & df['drive'].isnull() & df['size'].isnull() & df['type'].isnull() & df['paint_color'].isnull() & df['manufacturer'].isnull()  & df['cylinders'].isnull())]

### Modeling

With your (almost?) final dataset in hand, it is now time to build some models.  Here, you should build a number of different regression models with the price as the target.  In building your models, you should explore different parameters and be sure to cross-validate your findings.

In [ ]:
# Prepare all possible regression models for prediction and evaluation
all_models = [
    ('ridge',Ridge()),
    ('lasso',Lasso()),
    ('elastic_net',ElasticNet()),
    ('linear_regression',LinearRegression())
]

In [ ]:
for model in all_models:
  print(model)

<built-in method index of tuple object at 0x7fcc6f9bc480>
<built-in method index of tuple object at 0x7fcc6f4d18c0>
<built-in method index of tuple object at 0x7fcc6fe92780>
<built-in method index of tuple object at 0x7fcc6f4d3540>


### Evaluation

With some modeling accomplished, we aim to reflect on what we identify as a high-quality model and what we are able to learn from this.  We should review our business objective and explore how well we can provide meaningful insight into drivers of used car prices.  Your goal now is to distill your findings and determine whether the earlier phases need revisitation and adjustment or if you have information of value to bring back to your client.

### Deployment

Now that we've settled on our models and findings, it is time to deliver the information to the client.  You should organize your work as a basic report that details your primary findings.  Keep in mind that your audience is a group of used car dealers interested in fine-tuning their inventory.

In [ ]:
# What features make a car more or less expensive .
# Provide clear recommedations to used car dealers about what customers value in a used car
#After understanding, preparing, and modeling your data, write up a basic report that details your primary findings. Your audience for this report is a group of used car dealers interested in fine-tuning their inventory.
# From existing experience , you can tailor your findings towards manufacturer's dealerships and private individual car dealers distinctively .

**Next steps with recommendations**

**Luxury tax** Some features of a car are region specific , for eg - In the UK, we are required to pay luxury tax on top of regular tax for each year for vehicles having RRP > 40K £ . Hence , a feature like “IsLuxTaxPayable” can be a critical determinant for some purchasers especially when even for the same car brand and model and same manufacturing year, a submodel can edge past the RRP beyond 40K threshold . This is called Luxury Trap here in the UK and adds to running costs of vehicles.
Other features of interest and in normal practice

1.**Front wheel drive** - Geography is important here as in hilly areas (such as Scotland which is also in UK), stability  of cars is more important hence 2.
**All-wheel-drive** (instead of FWD ) is more desirable to customers and often shoots up prices.

2.**Running Costs**- Insurance group taken into consideration for pricing vehicles because luxury cars often have higher premiums. Annual Servicing charges also are a factor with preium cars.

3.**Accident history of cars**(AccidentCategory -  S/N/NaN) - Whether the car had an accident category , whether the damage was structural or non-structural. significantly lowers the price relatively .

4.**Manufacturer approved** - Whether car dealer belongs to official manufacturers  dealerships such as BMW , Porsche or private dealers .

**Limitations**

**Generalization to new makes** When new car brands and models come out , there wouldnt be enough data to make accurate predictions as there would be many unknowns in the existing features.

**Time based sales** Car sales is also heavily impacted by seasonal variations including festivals and occassions such as year end when demand is high due to holiday season, hence the algo has to evolve to consider time series data for dealers to price vehicles in order to make more profits .

**Manufacturing warranty**- Customers increasingly value existing warranty offered by original manufacturer which normally lasts for 3 years and hence in spite of a general assumption that car depreciates heavily in the first 3 years , this is an important factor to consider as desirability of customers would be high for buying such cars.

**Consumer preference** The above dataset does not have any indicator for consumer preference hence additional segmentation data is required for brand conscious customers (and could afford) who would buy only elite brands vs medium-tier customers hence in real-world sophisticated models are needed tailored for different brands.

**Checklist for a proper submission**

Built baseline Linear Regression

Showed coefficient interpretation

Added polynomial terms

Compared Ridge vs Lasso

Did cross-validation properly

Used SFS or RFE

Explained bias–variance tradeoff

Discussed overfitting

And most importantly:

Explained results in business language.